## 0. Title, learning objectives, roadmap (3 min)

- **Course:** Scientific Programming for Physics (Oct 27, 2026)
- **Author:** Durdona Uskenbayeva
- **Advisor:** Aleksandr Kapitonov
- **Description:** Numerical solutions of Ordinary Differential Equations (ODEs), Euler vs. RK45 comparison, step-size convergence analysis, harmonic and Van der Pol oscillators.

**Learning Objectives:**
1. Understand explicit Euler integration and its limitations.
2. Compare custom numerical methods with SciPy adaptive solvers (`solve_ivp`).
3. Analyze step-size convergence and stability in differential equations.
4. Simulate second-order physical systems and nonlinear limit cycles.

## 1. Motivation (5 min)

In physics, almost every interesting system — from a swinging pendulum to planetary orbits or electrical circuits — is described by differential equations.

**The problem?** While textbook examples like $\frac{dy}{dt} = -2y$ have neat, exact formulas, real-world nonlinear systems rarely do. When paper-and-pencil math hits a wall, we turn to **numerical integration**.

Today, we won't just look at dry formulas. We will build a solver from scratch, intentionally **break it** to see where it fails, compare it with industry-standard SciPy tools, and simulate systems that loop endlessly in phase space.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# 1. Generalized Euler Method (Supports both scalar and vector systems)
def euler_method(f, t0, y0, h, t_end):
    t_values = np.arange(t0, t_end + h, h)
    y0 = np.atleast_1d(y0)
    y_values = np.zeros((len(t_values), len(y0)))
    y_values[0] = y0

    for i in range(1, len(t_values)):
        t = t_values[i - 1]
        y = y_values[i - 1]
        y_values[i] = y + h * np.array(f(t, y))

    return t_values, y_values

# Test differential equation: dy/dt = -2y
def dydt(t, y):
    return -2 * y

# Parameters
t0, y0, t_end, h = 0.0, 1.0, 5.0, 0.1
t_euler, y_euler = euler_method(dydt, t0, y0, h, t_end)
print("Generalized Euler method executed successfully!")

## 2. Physics background (10 min)
Governing ODE: $\frac{dy}{dt} = -2y$
Analytic solution: $y(t) = y_0 e^{-2t}$

## 3. Comparison with SciPy's RK45 solver (5 min)

In [ ]:
# Compare Euler method (with good and bad step sizes) against SciPy RK45
t_eval = np.linspace(t0, t_end, 200)
sol = solve_ivp(dydt, [t0, t_end], [y0], t_eval=t_eval)

# Case 1: Reasonable step size (h = 0.1)
t_e1, y_e1 = euler_method(dydt, t0, y0, 0.1, t_end)

# Case 2: Too large step size (h = 1.1) -> Causes numerical instability!
t_e2, y_e2 = euler_method(dydt, t0, y0, 1.1, t_end)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# Left plot: Normal behavior
ax1.plot(sol.t, sol.y[0], 'k-', label='Exact / RK45', linewidth=2)
ax1.plot(t_e1, y_e1, 'bo--', label='Euler (h = 0.1)', markersize=4)
ax1.set_title('Good Step Size: Tracks the Curve')
ax1.set_xlabel('Time (t)')
ax1.set_ylabel('y(t)')
ax1.legend()
ax1.grid(True, alpha=0.4)

# Right plot: Numerical explosion
ax2.plot(sol.t, sol.y[0], 'k-', label='Exact / RK45', linewidth=2)
ax2.plot(t_e2, y_e2, 'ro--', label='Euler (h = 1.1) - Unstable!', markersize=6)
ax2.set_title('Large Step Size: Method Breaks Down!')
ax2.set_xlabel('Time (t)')
ax2.legend()
ax2.grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

## 4. Convergence Analysis: Impact of Step Size (h) (7 min)

In [ ]:
# Convergence Analysis: Impact of step size (h) on Euler Method Error
step_sizes = [0.2, 0.1, 0.05, 0.02, 0.01]
max_errors = []

for h_val in step_sizes:
    t_vals, y_vals = euler_method(dydt, t0, y0, h_val, t_end)
    y_exact_vals = np.exp(-2 * t_vals)
    error = np.max(np.abs(y_vals - y_exact_vals))
    max_errors.append(error)

# Plotting the error convergence graph
plt.figure(figsize=(8, 5))
plt.plot(step_sizes, max_errors, 'ro-', label='Euler Method Error', linewidth=2, markersize=6)
plt.xlabel('Step Size (h)')
plt.ylabel('Maximum Absolute Error')
plt.title('Convergence Analysis: Error vs. Step Size (h)')
plt.legend()
plt.grid(True)
plt.show()

## 5. Case Study 2: Solving Second-Order Systems (Harmonic Oscillator) (10 min)

In [ ]:
# Harmonic Oscillator system with Energy Conservation Check
def harmonic_system(t, y):
    return np.array([y[1], -y[0]])

t0_h, t_end_h = 0.0, 20.0
y0_h = np.array([1.0, 0.0]) # position = 1.0, velocity = 0.0
h_h = 0.05
t_eval_h = np.arange(t0_h, t_end_h + h_h, h_h)
sol_h = solve_ivp(harmonic_system, [t0_h, t_end_h], y0_h, t_eval=t_eval_h, method='RK45')

# Calculate Total Energy: E = 0.5 * (velocity^2 + position^2)
positions = sol_h.y[0]
velocities = sol_h.y[1]
energy = 0.5 * (velocities**2 + positions**2)

# Plotting phase space and energy conservation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Position and Velocity over time
ax1.plot(sol_h.t, positions, 'b-', label='Position y(t)', linewidth=2)
ax1.plot(sol_h.t, velocities, 'r--', label='Velocity v(t)', linewidth=1.5)
ax1.set_xlabel('Time (t)')
ax1.set_ylabel('State Variables')
ax1.set_title('Harmonic Oscillator Trajectory')
ax1.legend()
ax1.grid(True)

# 2. Energy Conservation
ax2.plot(sol_h.t, energy, 'g-', label='Total Energy E(t)', linewidth=2)
ax2.set_xlabel('Time (t)')
ax2.set_ylabel('Energy')
ax2.set_title('Energy Conservation Test ($E = \\frac{1}{2}(v^2 + y^2)$)')
ax2.set_ylim(0.45, 0.55) # Zooming in to check stability
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

print(f"Initial Energy: {energy[0]:.5f}, Final Energy: {energy[-1]:.5f}")

## 6. Automated Tests (Required Verification) (5 min)

In [ ]:
# Automated verification tests

# 1. Test Euler method shape, length, and initial condition
t_test_e, y_test_e = euler_method(dydt, 0.0, 1.0, 0.1, 1.0)
assert len(t_test_e) == len(y_test_e), "Euler output arrays must have the same length!"
assert y_test_e[0] == 1.0, "Initial condition for Euler method is incorrect!"
assert y_test_e.ndim == 1 or y_test_e.shape[1] == 1, "Euler method should handle scalar/vector outputs properly!"

# 2. Test SciPy RK45 solver consistency using assert_allclose
sol_test = solve_ivp(dydt, [0.0, 1.0], [1.0], t_eval=[1.0])
expected_val = np.exp(-2.0 * 1.0)
np.testing.assert_allclose(sol_test.y[0][0], expected_val, rtol=1e-2,
                            err_msg="RK45 solution deviates too much from the exact value!")

# 3. Test Harmonic Oscillator shape and energy conservation bounds
assert sol_h.y.shape[0] == 2, "Harmonic oscillator must have 2 state variables (position and velocity)!"
energy_drift = np.max(np.abs(energy - energy[0]))
assert energy_drift < 0.01, "Energy conservation check failed: energy drift is too high!"

print("Tests passed: Euler shape, RK45 accuracy, and Energy conservation verified.")

## 7. Performance Comparison of Different Solvers (*RK45*, *Radau*, *BDF*) (5 min)

In [ ]:
# Stiff test problem (Robertson-like or stiff exponential decay)
def stiff_system(t, y):
    return np.array([-1000.0 * y[0]])

t0_stiff, t_end_stiff = 0.0, 1.0
y0_stiff = np.array([1.0])
t_eval_stiff = np.linspace(t0_stiff, t_end_stiff, 100)

# Comparing solvers on a stiff problem
sol_rk_stiff = solve_ivp(stiff_system, [t0_stiff, t_end_stiff], y0_stiff, method='RK45', t_eval=t_eval_stiff)
sol_radau = solve_ivp(stiff_system, [t0_stiff, t_end_stiff], y0_stiff, method='Radau', t_eval=t_eval_stiff)
sol_bdf = solve_ivp(stiff_system, [t0_stiff, t_end_stiff], y0_stiff, method='BDF', t_eval=t_eval_stiff)

print(f"Stiff Test Results:")
print(f"RK45 function evaluations: {sol_rk_stiff.nfev}")
print(f"Radau function evaluations: {sol_radau.nfev}")
print(f"BDF function evaluations: {sol_bdf.nfev}")

## 8. Advanced Application: Nonlinear Van der Pol Oscillator (5 min)

In [ ]:
def vanderpol(t, y, mu=1.5):
    return np.array([y[1], mu * (1.0 - y[0]**2) * y[1] - y[0]])

t_span = [0.0, 25.0]
t_vdp = np.linspace(0.0, 25.0, 800)

# Different starting points: one inside the cycle, two outside
initial_conditions = [
    ([0.2, 0.2], 'green', 'Start Inside (0.2, 0.2)'),
    ([3.0, 2.0], 'crimson', 'Start Outside (3.0, 2.0)'),
    ([-2.5, -2.0], 'royalblue', 'Start Outside (-2.5, -2.0)')
]

plt.figure(figsize=(8, 6))

for y0_init, color, label in initial_conditions:
    sol_vdp = solve_ivp(vanderpol, t_span, y0_init, t_eval=t_vdp)
    plt.plot(sol_vdp.y[0], sol_vdp.y[1], color=color, label=label, linewidth=1.8, alpha=0.85)
    # Mark the starting point with a dot
    plt.scatter(y0_init[0], y0_init[1], color=color, s=50, zorder=5)

plt.xlabel('Position y(t)')
plt.ylabel('Velocity v(t)')
plt.title('Van der Pol Phase Portrait: Everything Converges to the Limit Cycle')
plt.legend()
plt.grid(True, alpha=0.4)
plt.show()

## 9. Audience Exercise (10 min)
*Hands-on task cells for the audience to run and modify during the lecture.*
- **Easy Task:** Change the step size ($h$) in the Euler method from 0.1 to 0.05 and observe how the error changes on the graph.
- **Medium Task:** Complete the missing part of the harmonic oscillator step using the code cell below.
- **Stretch Task:** Try modifying the damping parameter $\mu$ in the Van der Pol oscillator and observe how the limit cycle shape changes.

In [ ]:
from ipywidgets import interact


@interact(mu=(0.1, 5.0, 0.2))
def interactive_vdp(mu=1.5):
    sol = solve_ivp(
        lambda t, y: vanderpol(t, y, mu=mu),
        [0, 30],
        [1.0, 0.0],
        t_eval=np.linspace(0, 30, 1000),
    )
    plt.figure(figsize=(7, 4.5))
    plt.plot(sol.y[0], sol.y[1], "darkorange", linewidth=2)
    plt.title(f"Interactive Van der Pol Limit Cycle (mu = {mu:.1f})")
    plt.xlabel("Position y(t)")
    plt.ylabel("Velocity v(t)")
    plt.xlim(-3, 3)
    plt.ylim(-8, 8)
    plt.grid(True)
    plt.show()

## 10. Key Takeaways & Discussion (5 min)

* **Simple isn't always safe:** The Euler method is easy to write in 5 lines of code, but picking a slightly large step size ($h$) ruins the physics completely.
* **Let the solver adapt:** Tools like `RK45` adjust their own step size automatically, keeping energy conserved in oscillating systems.
* **Match the tool to the physics:** Fast-changing ("stiff") equations need implicit solvers like `Radau` or `BDF` to avoid wasting computation time.
* **Phase space reveals hidden order:** Even when nonlinear equations look messy over time, plotting position vs. velocity uncovers clean geometric structures like limit cycles.

**Questions for discussion:**
1. Where in real life (engineering, biology, or electronics) have you seen systems that behave like a Van der Pol limit cycle?
2. Why didn't energy stay 100% constant in our harmonic oscillator test, even with `RK45`?